# KG Entity Lookup & Candidate Search
## Wikidata · DBpedia — Musical Entity Lookup

This notebook **searches** Wikidata and DBpedia for candidate URIs for the following entity types used in `MusicRecSyst.ttl`:

| Section | Entities |
|---------|----------|
| **Decades** | 1960s, 1970s, 1980s, 1990s, 2000s, 2010s, 2020s |
| **Pitch Classes** | C, C#, D, D#, E, F, F#, G, G#, A, A#, B |
| **Musical Notes** | C4 (Dó), D4 (Ré), … (absolute MIDI pitch names) |
| **Keys** | C major, A minor, G major, … |
| **Modes** | major, minor, dorian, phrygian, … |
| **Genres** | + parent / child genre hierarchy |
| **Instruments** | + subclass / part-of hierarchy |

> **⚠ This notebook only searches and displays candidates — no TTL is written.**  
> Review the results and decide which URIs to add to `MusicRecSyst.ttl` manually.

## 📦 1 — Install & Import Dependencies

In [19]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["requests", "SPARQLWrapper", "pandas", "tabulate"]:
    pip_install(pkg)

print("✅ All dependencies ready.")


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


✅ All dependencies ready.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [20]:
import requests
import pandas as pd
import time
import json
from SPARQLWrapper import SPARQLWrapper, JSON
from IPython.display import display, HTML, Markdown

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

# ── Endpoint config ────────────────────────────────────────────────────────────
WD_SPARQL   = "https://query.wikidata.org/sparql"
DBP_SPARQL  = "https://dbpedia.org/sparql"
WD_LOOKUP   = "https://www.wikidata.org/w/api.php"
DBP_LOOKUP  = "https://lookup.dbpedia.org/api/search"

HEADERS = {"User-Agent": "SONATA-KG-Lookup/1.0 (research notebook)"}

# ── Helpers ────────────────────────────────────────────────────────────────────

def wd_sparql(query: str, delay: float = 0.5) -> pd.DataFrame:
    """Run a SPARQL query against Wikidata and return a DataFrame."""
    time.sleep(delay)
    sp = SPARQLWrapper(WD_SPARQL)
    sp.addCustomHttpHeader("User-Agent", HEADERS["User-Agent"])
    sp.setQuery(query)
    sp.setReturnFormat(JSON)
    try:
        res = sp.query().convert()
        rows = res["results"]["bindings"]
        if not rows:
            return pd.DataFrame()
        return pd.DataFrame([{k: v["value"] for k, v in r.items()} for r in rows])
    except Exception as e:
        print(f"  ⚠ Wikidata SPARQL error: {e}")
        return pd.DataFrame()


def dbp_sparql(query: str, delay: float = 0.5) -> pd.DataFrame:
    """Run a SPARQL query against DBpedia and return a DataFrame."""
    time.sleep(delay)
    sp = SPARQLWrapper(DBP_SPARQL)
    sp.addCustomHttpHeader("User-Agent", HEADERS["User-Agent"])
    sp.setQuery(query)
    sp.setReturnFormat(JSON)
    try:
        res = sp.query().convert()
        rows = res["results"]["bindings"]
        if not rows:
            return pd.DataFrame()
        return pd.DataFrame([{k: v["value"] for k, v in r.items()} for r in rows])
    except Exception as e:
        print(f"  ⚠ DBpedia SPARQL error: {e}")
        return pd.DataFrame()


def wd_lookup(term: str, type_qid: str = None, limit: int = 5) -> pd.DataFrame:
    """Wikidata label lookup (wbsearchentities API)."""
    params = {"action": "wbsearchentities", "search": term,
              "language": "en", "format": "json", "limit": limit}
    try:
        r = requests.get(WD_LOOKUP, params=params, headers=HEADERS, timeout=10)
        hits = r.json().get("search", [])
        rows = [{"label": h.get("label",""), "description": h.get("description",""),
                 "wd_uri": f"http://www.wikidata.org/entity/{h['id']}",
                 "qid": h["id"]} for h in hits]
        df = pd.DataFrame(rows)
        if type_qid and not df.empty:
            df["type_filter"] = type_qid   # informational only
        return df
    except Exception as e:
        print(f"  ⚠ Wikidata lookup error for '{term}': {e}")
        return pd.DataFrame()


def dbp_lookup(term: str, type_uri: str = None, limit: int = 5) -> pd.DataFrame:
    """DBpedia lookup API search."""
    params = {"query": term, "format": "json", "maxResults": limit}
    if type_uri:
        params["typeName"] = type_uri
    try:
        r = requests.get(DBP_LOOKUP, params=params, headers=HEADERS, timeout=10)
        hits = r.json().get("docs", [])
        rows = [{"label": h.get("label", [""])[0],
                 "comment": h.get("comment", [""])[0][:100] if h.get("comment") else "",
                 "dbp_uri": h.get("resource", [""])[0],
                 "type": ", ".join(h.get("type", [])[:2])} for h in hits]
        return pd.DataFrame(rows)
    except Exception as e:
        print(f"  ⚠ DBpedia lookup error for '{term}': {e}")
        return pd.DataFrame()


def show(title: str, df: pd.DataFrame, source: str = ""):
    """Pretty-print a result DataFrame."""
    icon = "🔵" if "Wikidata" in source else "🟠" if "DBpedia" in source else "📋"
    display(Markdown(f"### {icon} {title} — {source}"))
    if df.empty:
        display(Markdown("_No results found._"))
    else:
        display(df.reset_index(drop=True))

print("✅ Helper functions loaded.")

✅ Helper functions loaded.


---
## 🗓️ 2 — Decades

Target: verify `wd:Q39911` (decade class) and find Q-items for each decade used in the dataset (1960s–2020s) so they can be added as named individuals in `MusicRecSyst.ttl`.

In [21]:
# ── 2a  Wikidata SPARQL: all decades (instances of Q39911) ──────────────────
DECADE_QUERY = """
SELECT ?decade ?decadeLabel ?start ?end ?prevLabel ?nextLabel WHERE {
  ?decade wdt:P31 wd:Q39911 .
  OPTIONAL { ?decade wdt:P580 ?start . }
  OPTIONAL { ?decade wdt:P582 ?end   . }
  OPTIONAL { ?decade wdt:P155 ?prev  . ?prev rdfs:label ?prevLabel FILTER(LANG(?prevLabel)="en") }
  OPTIONAL { ?decade wdt:P156 ?next  . ?next rdfs:label ?nextLabel FILTER(LANG(?nextLabel)="en") }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?start
"""

df_decades_wd = wd_sparql(DECADE_QUERY)
if not df_decades_wd.empty:
    # Keep decade URIs from the 1950s to 2030s to cover dataset range
    mask = df_decades_wd["decadeLabel"].str.contains(
        r"19[5-9]0s|200[0-9]s|201[0-9]s|2020s", na=False)
    df_decades_wd = df_decades_wd[mask].reset_index(drop=True)
show("Decades (1950s–2020s)", df_decades_wd, "Wikidata SPARQL")

  ⚠ Wikidata SPARQL error: HTTP Error 502: Bad Gateway


### 🔵 Decades (1950s–2020s) — Wikidata SPARQL

_No results found._

In [22]:
# ── 2b  DBpedia: decade resources ─────────────────────────────────────────────
decade_labels = ["1960s", "1970s", "1980s", "1990s", "2000s", "2010s", "2020s"]
rows = []
for d in decade_labels:
    df = dbp_lookup(d, limit=3)
    if not df.empty:
        df.insert(0, "decade", d)
        rows.append(df)

df_decades_dbp = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
show("Decades", df_decades_dbp, "DBpedia Lookup")

### 🟠 Decades — DBpedia Lookup

,decade,label,comment,dbp_uri,type
0,1960s,Soul music,African American community in the United States in the <B>1950s</B> and early <B>1960s</B>. It comb,http://dbpedia.org/resource/Soul_music,"http://dbpedia.org/ontology/MusicGenre, http://dbpedia.org/ontology/TopicalConcept"
1,1960s,Hard rock,"Hard rock, a loosely-defined subgenre of rock music, began in the mid-<B>1960s</B> with the garage",http://dbpedia.org/resource/Hard_rock,"http://dbpedia.org/ontology/MusicGenre, http://dbpedia.org/ontology/TopicalConcept"
2,1960s,Folk rock,"the United States, Canada, and the United Kingdom in the mid-<B>1960s</B>. In the U.S., folk rock e",http://dbpedia.org/resource/Folk_rock,"http://dbpedia.org/ontology/MusicGenre, http://dbpedia.org/ontology/TopicalConcept"
3,1970s,Soul music,African American community in the United States in the <B>1950s</B> and early <B>1960s</B>. It comb,http://dbpedia.org/resource/Soul_music,"http://dbpedia.org/ontology/MusicGenre, http://dbpedia.org/ontology/TopicalConcept"
4,1970s,Electronic music,"Electronic music is music that employs electronic musical instruments, digital instruments and circu",http://dbpedia.org/resource/Electronic_music,"http://dbpedia.org/ontology/MusicGenre, http://dbpedia.org/ontology/TopicalConcept"
5,1970s,Hard rock,"Hard rock, a loosely-defined subgenre of rock music, began in the mid-<B>1960s</B> with the garage",http://dbpedia.org/resource/Hard_rock,"http://dbpedia.org/ontology/MusicGenre, http://dbpedia.org/ontology/TopicalConcept"
6,1980s,Soul music,African American community in the United States in the <B>1950s</B> and early <B>1960s</B>. It comb,http://dbpedia.org/resource/Soul_music,"http://dbpedia.org/ontology/MusicGenre, http://dbpedia.org/ontology/TopicalConcept"
7,1980s,Alternative rock,"became widely popular in the <B>1980s</B>. In this instance, the word ""alternative"" refers to the g",http://dbpedia.org/resource/Alternative_rock,"http://dbpedia.org/ontology/MusicGenre, http://dbpedia.org/ontology/TopicalConcept"
8,1980s,Hard rock,"Hard rock, a loosely-defined subgenre of rock music, began in the mid-<B>1960s</B> with the garage",http://dbpedia.org/resource/Hard_rock,"http://dbpedia.org/ontology/MusicGenre, http://dbpedia.org/ontology/TopicalConcept"
9,1990s,Alternative rock,used to categorize rock music that emerged from the independent music underground of the <B>1970s</,http://dbpedia.org/resource/Alternative_rock,"http://dbpedia.org/ontology/MusicGenre, http://dbpedia.org/ontology/TopicalConcept"


---
## 🎼 3 — Pitch Classes

Twelve pitch classes (C through B) correspond to the chromatic scale. Looking up `wd:Q1848784` (pitch class) instances and DBpedia resources.

In [23]:
# ── 3a  Wikidata SPARQL: pitch class instances ────────────────────────────────
PC_QUERY = """
SELECT ?pc ?pcLabel ?semitone ?altLabel WHERE {
  ?pc wdt:P31 wd:Q1848784 .          # instance of: pitch class
  OPTIONAL { ?pc wdt:P1402 ?semitone . }   # semitone index (if present)
  OPTIONAL { ?pc skos:altLabel ?altLabel FILTER(LANG(?altLabel)="en") }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?semitone
"""
df_pc_wd = wd_sparql(PC_QUERY)
show("Pitch Classes", df_pc_wd, "Wikidata SPARQL")

### 🔵 Pitch Classes — Wikidata SPARQL

_No results found._

In [24]:
# ── 3b  DBpedia: pitch class resources ────────────────────────────────────────
PC_DBP_QUERY = """
SELECT DISTINCT ?pc ?label ?abstract WHERE {
  ?pc rdf:type dbo:PitchClass .
  ?pc rdfs:label ?label FILTER(LANG(?label) = "en") .
  OPTIONAL { ?pc dbo:abstract ?abstract FILTER(LANG(?abstract) = "en") }
}
ORDER BY ?label
"""
df_pc_dbp = dbp_sparql(PC_DBP_QUERY)

# Fallback: lookup API if SPARQL returns nothing
if df_pc_dbp.empty:
    pitch_names = ["C pitch class", "D pitch class", "E pitch class",
                   "F pitch class", "G pitch class", "A pitch class", "B pitch class"]
    rows = []
    for p in pitch_names:
        df = dbp_lookup(p, limit=2)
        if not df.empty:
            df.insert(0, "query", p)
            rows.append(df)
    df_pc_dbp = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

show("Pitch Classes", df_pc_dbp, "DBpedia")

### 🟠 Pitch Classes — DBpedia

,query,label,comment,dbp_uri,type
0,C pitch class,<B>Pitch</B> <B>class</B>,"apart, e.g., the <B>pitch</B> <B>class</B> <B>C</B> <B>consists</B> of the Cs in all octaves. ""The",http://dbpedia.org/resource/Pitch_class,
1,C pitch class,Set (music),"A set (<B>pitch</B> set, <B>pitch</B>-<B>class</B> set, set <B>class</B>, set form, set genus, <B>pi",http://dbpedia.org/resource/Set_(music),
2,D pitch class,Set (music),"A set (<B>pitch</B> set, <B>pitch</B>-<B>class</B> set, set <B>class</B>, set form, set genus, <B>pi",http://dbpedia.org/resource/Set_(music),
3,D pitch class,<B>Pitch</B> interval,"<B>pitch</B> from another, upward or <B>downward</B>. They are notated as follows: PI(a,b) = b − a",http://dbpedia.org/resource/Pitch_interval,
4,E pitch class,Insect,"arthropod phylum. Definitions and circumscriptions vary; usually, insects comprise a <B>class</B> w",http://dbpedia.org/resource/Insect,
5,E pitch class,Set theory (music),"be applied to tonal and atonal styles in any <B>equal</B> temperament tuning system, and to some <B",http://dbpedia.org/resource/Set_theory_(music),
6,F pitch class,Set (music),"A set (pitch set, pitch-class set, set class, set form, set genus, pitch collection) in music theory",http://dbpedia.org/resource/Set_(music),
7,F pitch class,Insect,"arthropod phylum. Definitions and circumscriptions vary; usually, insects comprise a <B>class</B> w",http://dbpedia.org/resource/Insect,
8,G pitch class,Set (music),"A set (<B>pitch</B> set, <B>pitch</B>-<B>class</B> set, set <B>class</B>, set form, set <B>genus</B>",http://dbpedia.org/resource/Set_(music),
9,G pitch class,Insect,"arthropod phylum. Definitions and circumscriptions vary; usually, insects comprise a <B>class</B> w",http://dbpedia.org/resource/Insect,


---
## 🎵 4 — Musical Notes (Absolute Pitch)

Individual note instances with specific octave (e.g. C4 = middle C = Dó). Looking up `wd:Q18373` (musical note) and solfège names.

In [25]:
# ── 4a  Wikidata SPARQL: musical note instances ───────────────────────────────
NOTE_QUERY = """
SELECT ?note ?noteLabel ?midiPitch ?solfege ?frequency WHERE {
  ?note wdt:P31 wd:Q18373 .          # instance of: musical note
  OPTIONAL { ?note wdt:P1402 ?midiPitch . }
  OPTIONAL { ?note wdt:P1812 ?solfege . }     # solfège name
  OPTIONAL { ?note wdt:P2098 ?frequency . }   # frequency (Hz)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?midiPitch
LIMIT 88
"""
df_notes_wd = wd_sparql(NOTE_QUERY)
show("Musical Notes", df_notes_wd, "Wikidata SPARQL")

# ── Also lookup the class itself ──────────────────────────────────────────────
df_note_class = wd_lookup("musical note", limit=5)
show("'musical note' — class lookup", df_note_class, "Wikidata Label API")

### 🔵 Musical Notes — Wikidata SPARQL

_No results found._

### 🔵 'musical note' — class lookup — Wikidata Label API

,label,description,wd_uri,qid
0,note sign,sign used in musical notation to describe a pitched sound (frequency + duration),http://www.wikidata.org/entity/Q263478,Q263478
1,🎵,Unicode character,http://www.wikidata.org/entity/Q87577161,Q87577161
2,musical note,heraldic figure,http://www.wikidata.org/entity/Q55400101,Q55400101
3,musical pitch,specific sound pitch defined by a pitch class and octave,http://www.wikidata.org/entity/Q137473569,Q137473569
4,♪,Unicode character,http://www.wikidata.org/entity/Q87526875,Q87526875


In [26]:
# ── 4b  DBpedia: musical notes + solfège names ───────────────────────────────
solfege_pairs = [
    ("C / Dó",  "C (musical note)"),
    ("D / Ré",  "D (musical note)"),
    ("E / Mi",  "E (musical note)"),
    ("F / Fá",  "F (musical note)"),
    ("G / Sol", "G (musical note)"),
    ("A / Lá",  "A (musical note)"),
    ("B / Si",  "B (musical note)"),
]
rows = []
for solf, term in solfege_pairs:
    df = dbp_lookup(term, limit=2)
    if not df.empty:
        df.insert(0, "note", solf)
        rows.append(df)
df_notes_dbp = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
show("Musical Notes", df_notes_dbp, "DBpedia Lookup")

### 🟠 Musical Notes — DBpedia Lookup

,note,label,comment,dbp_uri,type
0,C / Dó,Keyboard instrument,"A keyboard instrument is a <B>musical</B> instrument played using a keyboard, a row of levers which",http://dbpedia.org/resource/Keyboard_instrument,
1,C / Dó,<B>C</B> (<B>musical</B> <B>note</B>),"(the relative minor of <B>C</B> major), and the fourth <B>note</B> (F, A, B, <B>C</B>) of the Guido",http://dbpedia.org/resource/C_(musical_note),
2,D / Ré,<B>D</B>♭ (<B>musical</B> <B>note</B>),<B>D</B>♭ (<B>D</B>-flat) is a <B>musical</B> <B>note</B> lying a <B>diatonic</B> semitone above C a,http://dbpedia.org/resource/D♭_(musical_note),
3,D / Ré,<B>D</B>♯ (<B>musical</B> <B>note</B>),of the <B>D</B>♯ above middle C (or <B>D</B>♯4) is approximately 311.127 Hz. See pitch (<B>music</B,http://dbpedia.org/resource/D♯_(musical_note),
4,E / Mi,<B>E</B>♭ (<B>musical</B> <B>note</B>),"and a chromatic semitone below <B>E</B>, thus being <B>enharmonic</B> to D♯ (D-sharp) or re dièse.",http://dbpedia.org/resource/E♭_(musical_note),
5,E / Mi,<B>Musical</B> <B>note</B>,"In <B>music</B>, a <B>note</B> is a symbol denoting a <B>musical</B> sound. In <B>English</B> usage",http://dbpedia.org/resource/Musical_note,
6,F / Fá,Keyboard instrument,"A keyboard instrument is a <B>musical</B> instrument played using a keyboard, a row of levers which",http://dbpedia.org/resource/Keyboard_instrument,
7,F / Fá,<B>F</B>♯ (<B>musical</B> <B>note</B>),F♯ (F-sharp; also known as fa dièse or fi) is the seventh semitone of the solfège. It lies a chromat,http://dbpedia.org/resource/F♯_(musical_note),
8,G / Sol,<B>G</B>♯ (<B>musical</B> <B>note</B>),"G♯ (G-sharp) or sol dièse is the ninth semitone of the solfège. In the German pitch nomenclature, it",http://dbpedia.org/resource/G♯_(musical_note),
9,G / Sol,<B>G</B>♭ (<B>musical</B> <B>note</B>),"some temperaments, it is <B>not</B> the same as F♯. <B>G</B>♭ is a major third below B♭, whereas F♯",http://dbpedia.org/resource/G♭_(musical_note),


---
## 🎹 5 — Musical Keys

A key is a combination of pitch class + mode (e.g. C major, A minor). Wikidata models keys as `wd:Q848653` (key in music). DBpedia models them under `dbo:MusicalKey`.

In [27]:
# ── 5a  Wikidata SPARQL: musical key instances ────────────────────────────────
KEY_QUERY = """
SELECT ?key ?keyLabel ?tonicLabel ?modeLabel WHERE {
  ?key wdt:P31 wd:Q848653 .          # instance of: key (music)
  OPTIONAL { ?key wdt:P5771 ?tonic . ?tonic rdfs:label ?tonicLabel FILTER(LANG(?tonicLabel)="en") }
  OPTIONAL { ?key wdt:P6242 ?mode  . ?mode  rdfs:label ?modeLabel  FILTER(LANG(?modeLabel)="en")  }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?keyLabel
LIMIT 60
"""
df_keys_wd = wd_sparql(KEY_QUERY)
show("Musical Keys", df_keys_wd, "Wikidata SPARQL")

# Also confirm the class
df_key_class = wd_lookup("key music", limit=5)
show("'key (music)' — class lookup", df_key_class, "Wikidata Label API")

### 🔵 Musical Keys — Wikidata SPARQL

_No results found._

### 🔵 'key (music)' — class lookup — Wikidata Label API

_No results found._

In [28]:
# ── 5b  DBpedia SPARQL: musical keys ─────────────────────────────────────────
KEY_DBP_QUERY = """
SELECT DISTINCT ?key ?label WHERE {
  ?key rdf:type dbo:MusicalKey .
  ?key rdfs:label ?label FILTER(LANG(?label) = "en")
}
ORDER BY ?label
LIMIT 50
"""
df_keys_dbp = dbp_sparql(KEY_DBP_QUERY)

if df_keys_dbp.empty:
    key_terms = ["C major", "G major", "D major", "A major",
                 "A minor", "E minor", "D minor", "C minor"]
    rows = []
    for k in key_terms:
        df = dbp_lookup(k, limit=2)
        if not df.empty:
            df.insert(0, "key", k)
            rows.append(df)
    df_keys_dbp = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

show("Musical Keys", df_keys_dbp, "DBpedia")

### 🟠 Musical Keys — DBpedia

,key,label,comment,dbp_uri,type
0,C major,<B>Calgary</B>,–United States border. The <B>city</B> anchors the south end of the Statistics <B>Canada</B>-defined,http://dbpedia.org/resource/Calgary,"http://dbpedia.org/ontology/Settlement, http://dbpedia.org/ontology/City"
1,C major,John <B>Major</B>,United Kingdom and Leader of the <B>Conservative</B> Party from 1990 to 1997. <B>Major</B> was Fore,http://dbpedia.org/resource/John_Major,"http://dbpedia.org/ontology/Politician, http://dbpedia.org/ontology/Person"
2,G major,<B>Major</B> <B>general</B>,"Major general (abbreviated MG, maj. gen. and similar) is a military rank used in many countries. It",http://dbpedia.org/resource/Major_general,
3,G major,<B>Major</B> <B>general</B> (United States),"is a two-star <B>general</B> officer rank, with the pay <B>grade</B> of O-8. <B>Major</B> <B>genera",http://dbpedia.org/resource/Major_general_(United_States),
4,D major,<B>Major</B>,"<B>Major</B> is a military rank of commissioned officer status, with corresponding ranks existing in",http://dbpedia.org/resource/Major,
5,D major,<B>Major</B> general,is <B>derived</B> from the older rank of sergeant <B>major</B> general. The <B>disappearance</B> of,http://dbpedia.org/resource/Major_general,
6,A major,<B>Major</B>,"<B>Major</B> is <B>a</B> military rank of commissioned officer status, with corresponding ranks exis",http://dbpedia.org/resource/Major,
7,A major,United States <B>Army</B>,The United States Army (USA) is the land warfare service branch of the United States Armed Forces. I,http://dbpedia.org/resource/United_States_Army,"http://dbpedia.org/ontology/Organisation, http://dbpedia.org/ontology/MilitaryUnit"
8,A minor,<B>Minor</B> League Baseball,Minor League Baseball (MiLB) is a hierarchy of professional baseball leagues in the Americas that co,http://dbpedia.org/resource/Minor_League_Baseball,"http://dbpedia.org/ontology/Organisation, http://dbpedia.org/ontology/SportsLeague"
9,A minor,Scotland,"Scotland (Scots: Scotland, Scottish Gaelic: Alba [ˈal̪ˠapə] ()) is a country that is part of the Uni",http://dbpedia.org/resource/Scotland,


---
## 🎶 6 — Musical Modes

Modes (major, minor, dorian, phrygian, …) are modelled as `wd:Q731978` (musical mode) instances in Wikidata.

In [29]:
# ── 6a  Wikidata SPARQL: musical mode instances ───────────────────────────────
MODE_QUERY = """
SELECT ?mode ?modeLabel ?intervalPattern ?altLabel WHERE {
  ?mode wdt:P31 wd:Q731978 .         # instance of: musical mode
  OPTIONAL { ?mode wdt:P1402 ?intervalPattern . }
  OPTIONAL { ?mode skos:altLabel ?altLabel FILTER(LANG(?altLabel)="en") }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?modeLabel
"""
df_modes_wd = wd_sparql(MODE_QUERY)
show("Musical Modes", df_modes_wd, "Wikidata SPARQL")

# Class lookup
df_mode_class = wd_lookup("musical mode", limit=5)
show("'musical mode' — class lookup", df_mode_class, "Wikidata Label API")

### 🔵 Musical Modes — Wikidata SPARQL

,mode,altLabel,modeLabel
0,http://www.wikidata.org/entity/Q255331,minor scale,Aeolian mode
1,http://www.wikidata.org/entity/Q960729,dorian,Dorian mode
2,http://www.wikidata.org/entity/Q960729,mode 1,Dorian mode
3,http://www.wikidata.org/entity/Q14192207,NaN,Hagiopolitan Octoechos
4,http://www.wikidata.org/entity/Q5959143,NaN,Hypoaeolian mode
5,http://www.wikidata.org/entity/Q5959655,NaN,Hypoionian mode
6,http://www.wikidata.org/entity/Q5959732,NaN,Hypolocrian mode
7,http://www.wikidata.org/entity/Q57976142,mode 8,Hypomixolydian mode
8,http://www.wikidata.org/entity/Q135105630,NaN,Kurdish maqam
9,http://www.wikidata.org/entity/Q749541,NaN,Locrian mode


### 🔵 'musical mode' — class lookup — Wikidata Label API

,label,description,wd_uri,qid
0,mode,each of the sequences ordered by the notes within an octave according to the different sequences of tones and semitones,http://www.wikidata.org/entity/Q731978,Q731978
1,modernism,changes in musical form during the early 20th Century,http://www.wikidata.org/entity/Q2426218,Q2426218
2,Musical mode and estimation of time,scientific article,http://www.wikidata.org/entity/Q33326445,Q33326445


In [30]:
# ── 6b  DBpedia SPARQL: musical modes ────────────────────────────────────────
MODE_DBP_QUERY = """
SELECT DISTINCT ?mode ?label ?abstract WHERE {
  ?mode dct:subject <http://dbpedia.org/resource/Category:Musical_modes> .
  ?mode rdfs:label ?label FILTER(LANG(?label)="en") .
  OPTIONAL { ?mode dbo:abstract ?abstract FILTER(LANG(?abstract)="en") }
}
ORDER BY ?label
LIMIT 30
"""
df_modes_dbp = dbp_sparql(MODE_DBP_QUERY)
if not df_modes_dbp.empty and "abstract" in df_modes_dbp.columns:
    df_modes_dbp["abstract"] = df_modes_dbp["abstract"].str[:120]
show("Musical Modes", df_modes_dbp, "DBpedia SPARQL")

### 🟠 Musical Modes — DBpedia SPARQL

_No results found._

---
## 🎸 7 — Genres & Genre Hierarchy

Searches genre class `wd:Q188451` (music genre) for the seed genres present in `MusicRecSyst.ttl`, then recursively fetches **parent** (`wdt:P279` subclass of) and **child** (`^wdt:P279`) genres up to 2 hops.

In [31]:
# ── Seed genres from MusicRecSyst.ttl (extend this list as needed) ───────────
SEED_GENRES = [
    "rock music", "pop music", "pop rock", "indie rock",
    "jazz", "blues", "classical music", "hip hop music",
    "electronic music", "folk music", "country music",
    "rhythm and blues", "soul music", "metal music",
    "punk rock", "alternative rock",
]

# ── 7a  Wikidata: confirm each seed genre URI ─────────────────────────────────
genre_seed_rows = []
for g in SEED_GENRES:
    df = wd_lookup(g, limit=3)
    if not df.empty:
        df.insert(0, "seed", g)
        # Filter to rows whose description mentions "genre" or "music"
        mask = df["description"].str.contains("genre|music", case=False, na=False)
        df = df[mask] if mask.any() else df.head(1)
        genre_seed_rows.append(df)
    time.sleep(0.2)

df_genres_seed = pd.concat(genre_seed_rows, ignore_index=True) if genre_seed_rows else pd.DataFrame()
show("Seed Genres — URI Candidates", df_genres_seed, "Wikidata Label API")

### 🔵 Seed Genres — URI Candidates — Wikidata Label API

,seed,label,description,wd_uri,qid
0,rock music,rock music,popular music genre,http://www.wikidata.org/entity/Q11399,Q11399
1,rock music,rock band,musical group playing rock music,http://www.wikidata.org/entity/Q5741069,Q5741069
2,pop music,pop music,genre of popular music,http://www.wikidata.org/entity/Q37073,Q37073
3,pop music,popular music,music genres distributed to large audiences and considered to have wide appeal,http://www.wikidata.org/entity/Q373342,Q373342
4,pop rock,pop rock,music genre,http://www.wikidata.org/entity/Q484641,Q484641
5,indie rock,indie rock,"genre of rock music, subgenre of alternative rock",http://www.wikidata.org/entity/Q183504,Q183504
6,jazz,jazz,musical style and genre,http://www.wikidata.org/entity/Q8341,Q8341
7,blues,blues,vocal and instrumental music form,http://www.wikidata.org/entity/Q9759,Q9759
8,classical music,classical music,broad tradition of Western art music,http://www.wikidata.org/entity/Q9730,Q9730
9,classical music,art music,"serious music, as opposed to popular or folk music; meta-genre covering global classical music developments",http://www.wikidata.org/entity/Q1583807,Q1583807


In [32]:
# ── 7b  Wikidata SPARQL: genre hierarchy (parent + children) ─────────────────
# Uses wdt:P279 (subclass of) to walk both directions

GENRE_HIER_QUERY = """
SELECT DISTINCT ?genre ?genreLabel ?parent ?parentLabel ?child ?childLabel WHERE {
  # Genres that are subclasses of music genre (Q188451), up to 2 hops
  ?genre wdt:P279* wd:Q188451 .
  OPTIONAL {
    ?genre wdt:P279 ?parent .
    ?parent wdt:P279* wd:Q188451 .
    SERVICE wikibase:label { bd:serviceParam wikibase:language "en".
      ?parent rdfs:label ?parentLabel . }
  }
  OPTIONAL {
    ?child wdt:P279 ?genre .
    SERVICE wikibase:label { bd:serviceParam wikibase:language "en".
      ?child rdfs:label ?childLabel . }
  }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?genreLabel ?parentLabel ?childLabel
LIMIT 300
"""
df_genre_hier_wd = wd_sparql(GENRE_HIER_QUERY)
show("Genre Hierarchy (parent / children)", df_genre_hier_wd, "Wikidata SPARQL")

### 🔵 Genre Hierarchy (parent / children) — Wikidata SPARQL

,genre,genreLabel,parent,parentLabel,child,childLabel
0,http://www.wikidata.org/entity/Q5475211,Shengqiang,http://www.wikidata.org/entity/Q20643324,opera genre,NaN,NaN
1,http://www.wikidata.org/entity/Q105635529,blues genre,http://www.wikidata.org/entity/Q188451,music genre,NaN,NaN
2,http://www.wikidata.org/entity/Q96200402,electronic music genre,http://www.wikidata.org/entity/Q188451,music genre,NaN,NaN
3,http://www.wikidata.org/entity/Q106621729,folk music genre,http://www.wikidata.org/entity/Q188451,music genre,NaN,NaN
4,http://www.wikidata.org/entity/Q21006644,fusion music genre,http://www.wikidata.org/entity/Q188451,music genre,NaN,NaN
5,http://www.wikidata.org/entity/Q3177825,game piece,http://www.wikidata.org/entity/Q188451,music genre,NaN,NaN
6,http://www.wikidata.org/entity/Q1521426,gharana,http://www.wikidata.org/entity/Q188451,music genre,NaN,NaN
7,http://www.wikidata.org/entity/Q96195030,jazz genre,http://www.wikidata.org/entity/Q188451,music genre,NaN,NaN
8,http://www.wikidata.org/entity/Q117021501,music by instrument,http://www.wikidata.org/entity/Q188451,music genre,NaN,NaN
9,http://www.wikidata.org/entity/Q188451,music genre,NaN,NaN,http://www.wikidata.org/entity/Q105635529,blues genre


In [33]:
# ── 7c  DBpedia SPARQL: genre hierarchy ───────────────────────────────────────
GENRE_DBP_QUERY = """
SELECT DISTINCT ?genre ?label ?broader ?broaderLabel ?narrower ?narrowerLabel WHERE {
  ?genre rdf:type dbo:MusicGenre .
  ?genre rdfs:label ?label FILTER(LANG(?label)="en") .
  OPTIONAL {
    ?genre dbo:musicSubgenre ?narrower .
    ?narrower rdfs:label ?narrowerLabel FILTER(LANG(?narrowerLabel)="en")
  }
  OPTIONAL {
    ?broader dbo:musicSubgenre ?genre .
    ?broader rdfs:label ?broaderLabel FILTER(LANG(?broaderLabel)="en")
  }
}
ORDER BY ?label
LIMIT 200
"""
df_genre_dbp = dbp_sparql(GENRE_DBP_QUERY)
show("Genre Hierarchy", df_genre_dbp, "DBpedia SPARQL")

### 🟠 Genre Hierarchy — DBpedia SPARQL

,genre,label,narrower,narrowerLabel,broader,broaderLabel
0,http://dbpedia.org/resource/2-step_garage,2-step garage,NaN,NaN,NaN,NaN
1,http://dbpedia.org/resource/20th-century_classical_music,20th-century classical music,NaN,NaN,NaN,NaN
2,http://dbpedia.org/resource/21st-century_classical_music,21st-century Western classical music,NaN,NaN,NaN,NaN
3,http://dbpedia.org/resource/21st-century_classical_music,21st-century classical music,NaN,NaN,NaN,NaN
4,http://dbpedia.org/resource/A_cappella,A cappella,http://dbpedia.org/resource/Barbershop_music,Barbershop music,NaN,NaN
5,http://dbpedia.org/resource/A_cappella,A cappella,http://dbpedia.org/resource/Collegiate_a_cappella,Collegiate a cappella,NaN,NaN
6,http://dbpedia.org/resource/A_cappella,A cappella,http://dbpedia.org/resource/Puirt_à_beul,Puirt à beul,NaN,NaN
7,http://dbpedia.org/resource/Aak,Aak,NaN,NaN,http://dbpedia.org/resource/Korean_court_music,Korean court music
8,http://dbpedia.org/resource/Abecedarian_hymn,Abecedarian hymn,NaN,NaN,NaN,NaN
9,http://dbpedia.org/resource/Abendmusik,Abendmusik,NaN,NaN,NaN,NaN


---
## 🎻 8 — Instruments & Instrument Hierarchy

Searches `wd:Q34379` (musical instrument) class and `mo:Instrument` for the instruments used in `MusicRecSyst.ttl` (guitar, violin, string instruments, etc.), then fetches `wdt:P279` parent/child class chains and `wdt:P31` instance-of links.

In [34]:
# ── Seed instruments from the graph visualization ────────────────────────────
SEED_INSTRUMENTS = [
    "guitar", "violin", "piano", "drums", "bass guitar",
    "cello", "trumpet", "saxophone", "flute", "synthesizer",
    "electric guitar", "acoustic guitar", "string instrument",
    "keyboard instrument", "percussion instrument", "wind instrument",
]

# ── 8a  Wikidata: confirm each instrument URI ─────────────────────────────────
instr_seed_rows = []
for instr in SEED_INSTRUMENTS:
    df = wd_lookup(instr, limit=3)
    if not df.empty:
        df.insert(0, "seed", instr)
        mask = df["description"].str.contains("instrument|music", case=False, na=False)
        df = df[mask] if mask.any() else df.head(1)
        instr_seed_rows.append(df)
    time.sleep(0.2)

df_instr_seed = pd.concat(instr_seed_rows, ignore_index=True) if instr_seed_rows else pd.DataFrame()
show("Seed Instruments — URI Candidates", df_instr_seed, "Wikidata Label API")

### 🔵 Seed Instruments — URI Candidates — Wikidata Label API

,seed,label,description,wd_uri,qid
0,guitar,guitar,fretted string instrument,http://www.wikidata.org/entity/Q6607,Q6607
1,guitar,guitarist,musician who plays the guitar,http://www.wikidata.org/entity/Q855091,Q855091
2,violin,violin,bowed string instrument,http://www.wikidata.org/entity/Q8355,Q8355
3,piano,piano,musical keyboard instrument,http://www.wikidata.org/entity/Q5994,Q5994
4,drums,drum kit,musical instrument made up of a group of percussion instruments,http://www.wikidata.org/entity/Q128309,Q128309
5,bass guitar,bass guitar,electric or acoustic bass instrument,http://www.wikidata.org/entity/Q46185,Q46185
6,bass guitar,bass guitar performance,type of musical performance,http://www.wikidata.org/entity/Q115160654,Q115160654
7,bass guitar,Bass Guitar,British music magazine,http://www.wikidata.org/entity/Q4867919,Q4867919
8,cello,cello,musical instrument,http://www.wikidata.org/entity/Q8371,Q8371
9,trumpet,trumpet,musical instrument,http://www.wikidata.org/entity/Q8338,Q8338


In [35]:
# ── 8b  Wikidata SPARQL: instrument hierarchy (subclass chain) ───────────────
INSTR_HIER_QUERY = """
SELECT DISTINCT ?inst ?instLabel ?parent ?parentLabel ?child ?childLabel WHERE {
  ?inst wdt:P279* wd:Q34379 .        # subclass* of: musical instrument
  OPTIONAL {
    ?inst wdt:P279 ?parent .
    ?parent wdt:P279* wd:Q34379 .
    SERVICE wikibase:label { bd:serviceParam wikibase:language "en".
      ?parent rdfs:label ?parentLabel . }
  }
  OPTIONAL {
    ?child wdt:P279 ?inst .
    SERVICE wikibase:label { bd:serviceParam wikibase:language "en".
      ?child rdfs:label ?childLabel . }
  }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
  # Filter to keep only relevant families to avoid enormous result sets
  FILTER EXISTS {
    ?inst wdt:P279* wd:Q34379 .
    ?inst rdfs:label ?lbl FILTER(LANG(?lbl)="en")
    FILTER(STRSTARTS(LCASE(?lbl), "str") || STRSTARTS(LCASE(?lbl), "key") ||
           STRSTARTS(LCASE(?lbl), "wind") || STRSTARTS(LCASE(?lbl), "perc") ||
           STRSTARTS(LCASE(?lbl), "brass") || STRSTARTS(LCASE(?lbl), "guitar") ||
           STRSTARTS(LCASE(?lbl), "violin") || STRSTARTS(LCASE(?lbl), "piano") ||
           STRSTARTS(LCASE(?lbl), "drum") || STRSTARTS(LCASE(?lbl), "synth"))
  }
}
ORDER BY ?instLabel ?parentLabel ?childLabel
LIMIT 250
"""
df_instr_hier_wd = wd_sparql(INSTR_HIER_QUERY)
show("Instrument Hierarchy (parent / children)", df_instr_hier_wd, "Wikidata SPARQL")

### 🔵 Instrument Hierarchy (parent / children) — Wikidata SPARQL

,inst,instLabel,parent,parentLabel,child,childLabel
0,http://www.wikidata.org/entity/Q5309083,Drum charts,http://www.wikidata.org/entity/Q34379,musical instrument,NaN,NaN
1,http://www.wikidata.org/entity/Q5309097,Drum sequencer,http://www.wikidata.org/entity/Q646683,music sequencer,http://www.wikidata.org/entity/Q9336300,Side Man
2,http://www.wikidata.org/entity/Q5616828,Guitar fiddle,http://www.wikidata.org/entity/Q34379,musical instrument,NaN,NaN
3,http://www.wikidata.org/entity/Q3236956,Guitaret,http://www.wikidata.org/entity/Q936762,lamellophone,NaN,NaN
4,http://www.wikidata.org/entity/Q25101248,Guitaro,http://www.wikidata.org/entity/Q34379,musical instrument,NaN,NaN
...,...,...,...,...,...,...
245,http://www.wikidata.org/entity/Q11404,drum,http://www.wikidata.org/entity/Q19588483,struck membranophone,http://www.wikidata.org/entity/Q11420300,uchiwa daiko
246,http://www.wikidata.org/entity/Q11404,drum,http://www.wikidata.org/entity/Q19588483,struck membranophone,http://www.wikidata.org/entity/Q60742599,yaogu
247,http://www.wikidata.org/entity/Q11404,drum,http://www.wikidata.org/entity/Q19588483,struck membranophone,http://www.wikidata.org/entity/Q191464,zerbaghali
248,http://www.wikidata.org/entity/Q128309,drum kit,http://www.wikidata.org/entity/Q133163,percussion instrument,http://www.wikidata.org/entity/Q5731863,Bombo de batería


In [36]:
# ── 8c  DBpedia SPARQL: instrument hierarchy ─────────────────────────────────
INSTR_DBP_QUERY = """
SELECT DISTINCT ?inst ?label ?broader ?broaderLabel WHERE {
  ?inst rdf:type dbo:Instrument .
  ?inst rdfs:label ?label FILTER(LANG(?label)="en") .
  OPTIONAL {
    ?inst dbo:type ?broader .
    ?broader rdfs:label ?broaderLabel FILTER(LANG(?broaderLabel)="en")
  }
}
ORDER BY ?label
LIMIT 150
"""
df_instr_dbp = dbp_sparql(INSTR_DBP_QUERY)

if df_instr_dbp.empty:
    instr_terms = ["string instrument", "keyboard instrument",
                   "wind instrument", "percussion instrument",
                   "guitar", "violin", "piano", "drums"]
    rows = []
    for t in instr_terms:
        df = dbp_lookup(t, limit=3)
        if not df.empty:
            df.insert(0, "query", t)
            rows.append(df)
    df_instr_dbp = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

show("Instruments & Families", df_instr_dbp, "DBpedia")

### 🟠 Instruments & Families — DBpedia

,inst,label
0,http://dbpedia.org/resource/BBC_Theatre_Organ,BBC Theatre Organ
1,http://dbpedia.org/resource/Danelectro_Dano_Pro,Danelectro Dano Pro
2,http://dbpedia.org/resource/Fender_Bass_V,Fender Bass V
3,http://dbpedia.org/resource/Fender_Coronado,Fender Coronado
4,http://dbpedia.org/resource/Fender_Esquire,Fender Esquire
5,http://dbpedia.org/resource/Fender_Jaguar,Fender Jaguar
6,http://dbpedia.org/resource/Fender_Jaguar_Baritone_Custom,Fender Jaguar Baritone Custom
7,http://dbpedia.org/resource/Fender_Telecaster_Deluxe,Fender Telecaster Deluxe
8,http://dbpedia.org/resource/Fender_Telecaster_Plus,Fender Telecaster Plus
9,http://dbpedia.org/resource/Fender_Urge_Bass,Fender Urge Bass


---
## 📤 9 — Export All Results to CSV

Export every result table as a CSV file in `~/Downloads/kg_lookup_results/` for offline review before editing the TTL.

In [37]:
import os

OUT_DIR = os.path.expanduser("~/Downloads/kg_lookup_results")
os.makedirs(OUT_DIR, exist_ok=True)

exports = {
    "decades_wikidata":          locals().get("df_decades_wd",      pd.DataFrame()),
    "decades_dbpedia":           locals().get("df_decades_dbp",     pd.DataFrame()),
    "pitch_classes_wikidata":    locals().get("df_pc_wd",           pd.DataFrame()),
    "pitch_classes_dbpedia":     locals().get("df_pc_dbp",          pd.DataFrame()),
    "notes_wikidata":            locals().get("df_notes_wd",        pd.DataFrame()),
    "notes_dbpedia":             locals().get("df_notes_dbp",       pd.DataFrame()),
    "keys_wikidata":             locals().get("df_keys_wd",         pd.DataFrame()),
    "keys_dbpedia":              locals().get("df_keys_dbp",        pd.DataFrame()),
    "modes_wikidata":            locals().get("df_modes_wd",        pd.DataFrame()),
    "modes_dbpedia":             locals().get("df_modes_dbp",       pd.DataFrame()),
    "genres_seed_wikidata":      locals().get("df_genres_seed",     pd.DataFrame()),
    "genres_hierarchy_wikidata": locals().get("df_genre_hier_wd",  pd.DataFrame()),
    "genres_hierarchy_dbpedia":  locals().get("df_genre_dbp",      pd.DataFrame()),
    "instruments_seed_wikidata": locals().get("df_instr_seed",     pd.DataFrame()),
    "instruments_hier_wikidata": locals().get("df_instr_hier_wd",  pd.DataFrame()),
    "instruments_hier_dbpedia":  locals().get("df_instr_dbp",      pd.DataFrame()),
}

for name, df in exports.items():
    if df is not None and not df.empty:
        path = os.path.join(OUT_DIR, f"{name}.csv")
        df.to_csv(path, index=False)
        print(f"  ✅ Saved {len(df):>4} rows → {path}")
    else:
        print(f"  ⚠  No data for {name}")

print(f"\n📁 All CSVs saved to: {OUT_DIR}")

  ⚠  No data for decades_wikidata
  ✅ Saved   21 rows → /home/pfanyka/Downloads/kg_lookup_results/decades_dbpedia.csv
  ⚠  No data for pitch_classes_wikidata
  ✅ Saved   14 rows → /home/pfanyka/Downloads/kg_lookup_results/pitch_classes_dbpedia.csv
  ⚠  No data for notes_wikidata
  ✅ Saved   14 rows → /home/pfanyka/Downloads/kg_lookup_results/notes_dbpedia.csv
  ⚠  No data for keys_wikidata
  ✅ Saved   16 rows → /home/pfanyka/Downloads/kg_lookup_results/keys_dbpedia.csv
  ✅ Saved   26 rows → /home/pfanyka/Downloads/kg_lookup_results/modes_wikidata.csv
  ⚠  No data for modes_dbpedia
  ✅ Saved   27 rows → /home/pfanyka/Downloads/kg_lookup_results/genres_seed_wikidata.csv
  ✅ Saved   25 rows → /home/pfanyka/Downloads/kg_lookup_results/genres_hierarchy_wikidata.csv
  ✅ Saved  200 rows → /home/pfanyka/Downloads/kg_lookup_results/genres_hierarchy_dbpedia.csv
  ✅ Saved   32 rows → /home/pfanyka/Downloads/kg_lookup_results/instruments_seed_wikidata.csv
  ✅ Saved  250 rows → /home/pfanyka/Downlo

---
## ✅ 10 — Review Checklist

After running all sections, use this checklist to decide what to add to `MusicRecSyst.ttl`:

| Entity type | Suggested TTL action |
|-------------|----------------------|
| **Decades** | Add named individuals (e.g. `wd:Q34653`) with `wdt:P31`, `wdt:P361`, `wdt:P155/P156` triples |
| **Pitch classes** | Add Wikidata Q-items as `owl:sameAs` targets on a new `mrc:PitchClass` class, or reuse `wd:Q1848784` instances directly |
| **Musical notes** | Reuse `wd:Q18373` instances via `owl:sameAs` if relevant to key/mode properties |
| **Keys** | Reuse `wd:Q848653` instances; link via `owl:sameAs` on `mrc:Key` or `mo:Key` individuals |
| **Modes** | Reuse `wd:Q731978` instances; link on `mrc:Mode` class individuals |
| **Genres** | Add `owl:sameAs` to confirmed Q-items; add `rdfs:subClassOf wd:Q37073 / wd:Q11399` chains |
| **Instruments** | Add `owl:sameAs` to confirmed Q-items; build `rdfs:subClassOf mo:Instrument` hierarchy |

> **💡 Tip:** Copy the `wd_uri` value from any result row and paste it as `owl:sameAs <...>` in the relevant class or individual block in `MusicRecSyst.ttl`.